# OCR Pipeline 
This notebook extracts ingredients from skincare product label images using EasyOCR.

## Steps
1. Read image using EasyOCR
2. Extract and clean raw text
3. Split into individual ingredients
4. Return clean ingredient list

In [4]:
import easyocr
import re

reader = easyocr.Reader(['en'])

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [5]:
def extract_ingredients(image_path, confidence_threshold=0.3):
    
    # Step 1: OCR
    result = reader.readtext(image_path)
    lines = []
    for (bbox, text, confidence) in result:
        if confidence > confidence_threshold:
            lines.append(text)
    
    # Step 2: Join and basic cleaning
    raw_text = ' '.join(lines)
    raw_text = raw_text.replace(';', ',')
    raw_text = raw_text.replace('2O', '20')
    raw_text = raw_text.replace('_', '')
    
    # Step 3: Find ingredients section
    raw_lower = raw_text.lower()
    if 'ingredient' in raw_lower:
        start = raw_lower.find('ingredient')
        start = raw_text.find(',', start)
        raw_text = raw_text[start+1:]
    
    # Step 4: Split by comma
    ingredients = raw_text.split(',')
    
    # Step 5: Clean each ingredient
    cleaned = []
    for ingredient in ingredients:
        ingredient = ingredient.strip()
        ingredient = ingredient.strip('.')
        ingredient = re.sub(r'^\d+\s*', '', ingredient)
        ingredient = re.sub(r'\s+', ' ', ingredient).strip()
        ingredient = ingredient.lower()
        if len(ingredient) > 2:
            cleaned.append(ingredient)
    
    # Step 6: Fix full stop merges
    further_cleaned = []
    for ingredient in cleaned:
        if '.' in ingredient:
            parts = ingredient.split('.')
            for part in parts:
                part = part.strip()
                if len(part) > 2:
                    further_cleaned.append(part)
        else:
            further_cleaned.append(ingredient)
    
    # Step 7: Fix ceramide type merges
    final_ingredients = []
    for ingredient in further_cleaned:
        if 'ceramide' in ingredient and ingredient.count('ceramide') > 1:
            parts = re.split(r'(?<!^)(?=ceramide)', ingredient)
            for part in parts:
                part = part.strip()
                if len(part) > 2:
                    final_ingredients.append(part)
        else:
            final_ingredients.append(ingredient)
    
    return final_ingredients

In [7]:
ingredients = extract_ingredients('C:/Users/anamt/OneDrive/Desktop/skincare-analyzer/test_images/test1.jpg')
print(f"Total: {len(ingredients)}")
for i, ing in enumerate(ingredients, 1):
    print(f"{i}. {ing}")

C:\Users\anamt\OneDrive\Desktop\skincare-analyzer\venv310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Total: 32
1. cyclopentasiloxane
2. octyl salicylate
3. niacinamide
4. silica
5. avobenzone
6. octocrylene
7. butylene glycol
8. glycerine
9. homosalate
10. tapioca starch
11. neopentyl glycol diheptanoate
12. amps/hema crosspolymer (and) c13-15 alkane (and) coco-glucoside
13. octyldodecanol
14. polysorbate-20
15. zinc oxide
16. titanium dioxide
17. oats extract
18. zinc pca
19. cetearyl olivate & sorbitan olivate
20. ceramide ap
21. ceramide np
22. ceramide eos
23. d-panthenol
24. centella asiatica extract
25. ethylhexyl glycerine
26. phenoxyethanol
27. coco-glucoside
28. xanthan gum
29. tocopherol acetate
30. hyaluronic acid
31. citric acid
32. aqua
